# Score and compare the BLIP-2 query experiment

This notebook evaluates saved predictions from `query_ablation.ipynb` using the official VQAv2 evaluator. It runs on the CPU and does not load BLIP-2 or require a CUDA/ROCm GPU. Use the project's `blip2` environment (or an equivalent environment with the scoring dependencies installed).

The experiment compares five nested random query subsets at K=8 and K=16, the original K=32 baseline, and repeated-query controls at K=64 and K=128. The latter are not independently learned larger bottlenecks.

The notebook preserves the original scoring and output schema, adds paired confidence-interval figures, answer-type analysis, and question-level changes relative to K=32. Each analysis is conditional on the selected experiment, with all conditions evaluated on exactly the same questions. No additional inference or leave-one-query-out experiment is required.

**Run order:** select a completed run, verify the pinned evaluator, score, calculate statistics, then export tables and figures. Smoke/pilot runs are for debugging and preliminary exploration; use the main stage for the research report.

## 1. Configuration and experiment selection

Set `STAGE` to `smoke`, `pilot`, or `main` to match the ablation run. Prefer specifying `EXPERIMENT_DIR` using the path printed by the inference notebook. Relative paths are resolved from the project root. Alternatively, set `BLIP2_EXPERIMENT_DIR` before launching Jupyter.

If no directory is specified, the scorer searches completed runs for the chosen stage. `GPU_BACKEND` optionally filters to `cuda` or `rocm`; if more than one run matches, select the exact directory rather than silently mixing or choosing hardware. The backend refers to the machine that generated predictions, not the machine running this CPU notebook.

In [ ]:
from pathlib import Path
import copy
import hashlib
import json
import os
import types
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from filelock import FileLock
from IPython.display import display

STAGE = os.environ.get("BLIP2_STAGE", "main")
EXPERIMENT_DIR = os.environ.get("BLIP2_EXPERIMENT_DIR") or None
# Example: EXPERIMENT_DIR = "results/query_ablation/main-<signature>"
GPU_BACKEND = None  # Optional: "cuda" or "rocm" when searching for a run.
BOOTSTRAP_REPLICATES = 2000
BOOTSTRAP_SEED = 425

if STAGE not in {"smoke", "pilot", "main"}:
    raise ValueError("STAGE must be smoke, pilot, or main.")
if GPU_BACKEND not in {None, "cuda", "rocm"}:
    raise ValueError("GPU_BACKEND must be None, cuda, or rocm.")
if BOOTSTRAP_REPLICATES < 2:
    raise ValueError("At least two bootstrap replicates are required.")

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "scripts" else cwd
RESULTS_ROOT = PROJECT_ROOT / "results" / "query_ablation"


def read_json(path):
    return Path(path).read_text(encoding="utf-8")


def load_json(path):
    return json.loads(read_json(path))


def recorded_backend(experiment):
    # Older AMD manifests lack gpu_backend but record the ROCm version.
    if experiment.get("gpu_backend") in {"cuda", "rocm"}:
        return experiment["gpu_backend"]
    if experiment.get("rocm_version"):
        return "rocm"
    if experiment.get("cuda_version"):
        return "cuda"
    return "unknown"


if EXPERIMENT_DIR is None:
    candidates = []
    for path in sorted(RESULTS_ROOT.glob(f"{STAGE}-*/complete.json")):
        candidate_manifest = load_json(path.parent / "manifest.json")
        candidate = candidate_manifest["experiment"]
        if candidate["stage"] == STAGE and (
            GPU_BACKEND is None or recorded_backend(candidate) == GPU_BACKEND
        ):
            candidates.append((path.parent, candidate))
    if not candidates:
        raise FileNotFoundError(f"No completed {STAGE} experiment matches the selection. Run query_ablation.ipynb first.")
    if len(candidates) > 1:
        options = "\n".join(f"  {path} | {exp['gpu_name']} | {recorded_backend(exp)}" for path, exp in candidates)
        raise ValueError("Multiple completed runs match. Set EXPERIMENT_DIR to one of:\n" + options)
    EXPERIMENT_DIR = candidates[0][0]

EXPERIMENT_DIR = (PROJECT_ROOT / Path(EXPERIMENT_DIR)).resolve()
manifest = load_json(EXPERIMENT_DIR / "manifest.json")
experiment = manifest["experiment"]
complete = load_json(EXPERIMENT_DIR / "complete.json")
expected_signature = hashlib.sha256(json.dumps(experiment, sort_keys=True).encode()).hexdigest()
if complete["signature"] != manifest["signature"] or expected_signature != manifest["signature"]:
    raise ValueError("Experiment manifest or completion signature differs.")
if experiment["stage"] != STAGE:
    raise ValueError(f"Selected run is {experiment['stage']}, but STAGE is {STAGE}.")
backend = recorded_backend(experiment)
if GPU_BACKEND is not None and backend != GPU_BACKEND:
    raise ValueError("The selected experiment does not match GPU_BACKEND.")

question_ids = experiment["question_ids"]
conditions = experiment["conditions"]
names = [condition["name"] for condition in conditions]
if not question_ids or len(set(question_ids)) != len(question_ids):
    raise ValueError("Manifest question IDs must be nonempty and unique.")
if len(set(names)) != len(names) or names.count("k032") != 1:
    raise ValueError("Expected unique conditions and a single k032 baseline.")
expected_counts = {8: len(experiment["query_seeds"]), 16: len(experiment["query_seeds"]), 32: 1, 64: 1, 128: 1}
if pd.Series([c["k"] for c in conditions]).value_counts().to_dict() != expected_counts:
    raise ValueError("Unexpected query-count conditions in the manifest.")
if complete.get("condition_count", len(conditions)) != len(conditions) or complete.get("questions_per_condition", len(question_ids)) != len(question_ids):
    raise ValueError("Completion counts differ from the manifest.")

print(f"Scoring: {EXPERIMENT_DIR}")
print(f"{experiment['gpu_name']} ({backend}) | {len(conditions)} conditions × {len(question_ids):,} questions")

## 2. Official VQAv2 evaluator

Download the evaluator from GT-Vision-Lab at a pinned revision on the first run, verify its SHA256, and reuse the local copy afterward. Only its Python 2 print syntax is updated for Python 3; the answer normalization and ten-annotator consensus scoring are unchanged. Source: https://github.com/GT-Vision-Lab/VQA.

The evaluator is loaded from a verified text file, so no separate `.py` project script is necessary.

In [ ]:
EVALUATOR_REVISION = "a013f0043c1e2cdc995922dfe257f7149aa9af06"
EVALUATOR_SHA256 = "f08edfcad5be0112500993e245c706b6cb928eadebe203f89f838e5e0d04bec8"
EVALUATOR_URL = (
    f"https://raw.githubusercontent.com/GT-Vision-Lab/VQA/{EVALUATOR_REVISION}/"
    "PythonEvaluationTools/vqaEvaluation/vqaEval.py"
)
cache = RESULTS_ROOT / "evaluator"
cache.mkdir(parents=True, exist_ok=True)
source_path = cache / f"vqaEval-{EVALUATOR_REVISION}.txt"
with FileLock(str(cache / "download.lock"), timeout=60):
    if source_path.exists():
        source_bytes = source_path.read_bytes()
    else:
        with urllib.request.urlopen(EVALUATOR_URL, timeout=60) as response:
            source_bytes = response.read()
    if hashlib.sha256(source_bytes).hexdigest() != EVALUATOR_SHA256:
        raise ValueError("Official evaluator source failed its SHA256 check.")
    source_path.write_bytes(source_bytes)

source = source_bytes.decode("utf-8")
source = source.replace('print "computing accuracy"', 'print("computing accuracy")')
source = source.replace('print "Done computing accuracy"', 'print("Done computing accuracy")')
official_evaluator = types.ModuleType("official_vqa_evaluator")
exec(compile(source, str(source_path), "exec"), official_evaluator.__dict__)
VQAEval = official_evaluator.VQAEval
print("Official VQA evaluator verified:", EVALUATOR_REVISION)


## 3. Score saved predictions

Verify the exact annotation file and completed records against the manifest. Each condition must use the same ordered question IDs and image IDs. The official evaluator calculates per-question and aggregate VQA scores (percent, not fractions). Generation latency, throughput and GPU memory come from the original inference logs, not the current scoring machine.

Raw answers are retained unchanged for the later literal answer-agreement calculation.

In [ ]:
annotations_path = PROJECT_ROOT / "data" / "validation_dataset" / "v2_mscoco_val2014_annotations_subset.json"
if hashlib.sha256(annotations_path.read_bytes()).hexdigest() != experiment["annotation_file_sha256"]:
    raise ValueError("Annotations changed since predictions were generated.")
annotations = json.loads(annotations_path.read_text(encoding="utf-8"))["annotations"]
annotation_by_id = {int(annotation["question_id"]): annotation for annotation in annotations}
if len(annotation_by_id) != len(annotations) or not set(question_ids).issubset(annotation_by_id):
    raise ValueError("Missing or duplicate annotation IDs.")


def score_predictions(predictions):
    ids = [item["question_id"] for item in predictions]
    if ids != question_ids:
        raise ValueError("Prediction IDs must match the exact ordered evaluation set.")
    ground_truth = types.SimpleNamespace(
        qa={qid: copy.deepcopy(annotation_by_id[qid]) for qid in question_ids},
        getQuesIds=lambda: question_ids,
    )
    answers = types.SimpleNamespace(qa={item["question_id"]: copy.deepcopy(item) for item in predictions})
    evaluator = VQAEval(ground_truth, answers, n=6)
    evaluator.evaluate(quesIds=question_ids)
    return evaluator


summaries = []
scored_rows = []
answer_type_rows = []
# Read only completed records while holding the same lock used by generation.
with FileLock(str(EXPERIMENT_DIR / "experiment.lock"), timeout=0):
    if load_json(EXPERIMENT_DIR / "complete.json") != complete or load_json(EXPERIMENT_DIR / "manifest.json") != manifest:
        raise ValueError("Experiment changed after selection. Rerun the configuration cell.")
    for condition in conditions:
        directory = EXPERIMENT_DIR / condition["name"]
        status = json.loads((directory / "status.json").read_text(encoding="utf-8"))
        if status["state"] != "complete" or status["completed"] != len(question_ids):
            raise ValueError(f"Incomplete condition {condition['name']}.")
        records = [json.loads(line) for line in (directory / "records.jsonl").read_text(encoding="utf-8").splitlines()]
        if [row["question_id"] for row in records] != question_ids:
            raise ValueError("Missing, duplicate, or unexpected saved question IDs.")
        for row in records:
            if row["k"] != condition["k"] or row["query_seed"] != condition["seed"]:
                raise ValueError("Prediction condition does not match the manifest.")
            if row["image_id"] != int(annotation_by_id[row["question_id"]]["image_id"]):
                raise ValueError("Prediction image ID differs from its annotation.")
            if not isinstance(row["answer"], str):
                raise ValueError("Saved answer must be a string.")
        predictions = json.loads((directory / "predictions.json").read_text(encoding="utf-8"))
        if predictions != [{"question_id": row["question_id"], "answer": row["answer"]} for row in records]:
            raise ValueError("Exported predictions differ from the resumable records.")
        evaluator = score_predictions(predictions)
        frame = pd.DataFrame(records)
        metric_columns = ["generation_seconds", "peak_allocated_bytes", "peak_reserved_bytes", "generated_token_count"]
        values = frame[metric_columns].apply(pd.to_numeric, errors="raise")
        if not np.isfinite(values.to_numpy(dtype=float)).all() or (values < 0).any().any() or (values.generation_seconds <= 0).any():
            raise ValueError(f"Invalid generation metrics in {condition['name']}.")
        frame[metric_columns] = values
        summaries.append({
            "condition": condition["name"], "k": condition["k"], "seed": condition["seed"],
            "kind": condition["kind"], "questions": len(records),
            "vqa_accuracy_pct": evaluator.accuracy["overall"],
            "mean_latency_seconds": frame.generation_seconds.mean(),
            "median_latency_seconds": frame.generation_seconds.median(),
            "p95_latency_seconds": frame.generation_seconds.quantile(0.95),
            "throughput_questions_per_second": len(frame) / frame.generation_seconds.sum(),
            "peak_allocated_gib": frame.peak_allocated_bytes.max() / 2**30,
            "peak_reserved_gib": frame.peak_reserved_bytes.max() / 2**30,
            "mean_generated_tokens": frame.generated_token_count.mean(),
        })
        for row in records:
            row = dict(row, condition=condition["name"], kind=condition["kind"],
                       vqa_score_pct=evaluator.evalQA[row["question_id"]],
                       answer_type=annotation_by_id[row["question_id"]]["answer_type"])
            scored_rows.append(row)
        for answer_type, accuracy in evaluator.accuracy["perAnswerType"].items():
            answer_type_rows.append({"condition": condition["name"], "k": condition["k"],
                                     "seed": condition["seed"], "answer_type": answer_type,
                                     "vqa_accuracy_pct": accuracy,
                                     "questions": sum(annotation_by_id[qid]["answer_type"] == answer_type for qid in question_ids)})

summary = pd.DataFrame(summaries).sort_values(["k", "seed"])
per_question = pd.DataFrame(scored_rows)
if not per_question.vqa_score_pct.between(0, 100).all():
    raise ValueError("Official per-question scores must lie between 0 and 100.")
display(summary.round(4))


## 4. Paired statistical comparisons

Compare each condition with K=32 question by question. Resample unique images with replacement, preserving all questions about each sampled image. The bootstrap estimates a question-weighted mean difference; the 2.5th and 97.5th percentiles form a 95% interval. The same seed makes the resampling reproducible and paired across conditions.

For K=8 and K=16, average the five subset scores per question before bootstrapping the group difference. These intervals condition on the five observed subsets; they do not quantify variation over all possible query subsets. Report the between-subset sample SD separately. Intervals are pointwise, not adjusted for multiple comparisons. Differences are in percentage points (pp).

In [ ]:
def paired_image_bootstrap(delta, image_ids, replicates=BOOTSTRAP_REPLICATES):
    """Question-weighted mean difference, resampling distinct images together."""
    unique_images, codes = np.unique(np.asarray(image_ids), return_inverse=True)
    sums = np.bincount(codes, weights=np.asarray(delta, dtype=float))
    counts = np.bincount(codes)
    generator = np.random.default_rng(BOOTSTRAP_SEED)
    samples = np.empty(replicates)
    for index in range(replicates):
        drawn = generator.integers(0, len(unique_images), size=len(unique_images))
        samples[index] = sums[drawn].sum() / counts[drawn].sum()
    return np.quantile(samples, [0.025, 0.975])


scores = per_question.pivot(index="question_id", columns="condition", values="vqa_score_pct").reindex(question_ids)
if scores.isna().any().any():
    raise ValueError("All comparisons require the same complete question set.")
baseline = scores["k032"]
image_ids = [int(annotation_by_id[qid]["image_id"]) for qid in question_ids]
comparisons = []
for condition in experiment["conditions"]:
    delta = scores[condition["name"]] - baseline
    lower, upper = paired_image_bootstrap(delta, image_ids)
    comparisons.append({
        "condition": condition["name"], "k": condition["k"], "seed": condition["seed"],
        "delta_accuracy_pp": delta.mean(), "ci_lower_pp": lower, "ci_upper_pp": upper,
    })
comparisons = pd.DataFrame(comparisons).sort_values(["k", "seed"])

group_rows = []
for k, runs in summary.groupby("k", sort=True):
    names = runs.condition.tolist()
    mean_scores = scores[names].mean(axis=1)
    delta = mean_scores - baseline
    lower, upper = paired_image_bootstrap(delta, image_ids)
    group_rows.append({
        "k": k, "kind": runs.kind.iloc[0], "query_subsets": len(runs),
        "mean_accuracy_pct": mean_scores.mean(),
        "query_subset_accuracy_sd_pp": runs.vqa_accuracy_pct.std(ddof=1) if len(runs) > 1 else np.nan,
        "delta_vs_k32_pp": delta.mean(), "paired_ci_lower_pp": lower, "paired_ci_upper_pp": upper,
        "mean_median_latency_seconds": runs.median_latency_seconds.mean(),
        "mean_throughput_questions_per_second": runs.throughput_questions_per_second.mean(),
        "max_peak_allocated_gib": runs.peak_allocated_gib.max(),
        "mean_generated_tokens": runs.mean_generated_tokens.mean(),
    })
group_summary = pd.DataFrame(group_rows)
display(group_summary.round(4))



## 5. Answer-type and question-level analysis

VQAv2 labels answers as yes/no, number, or other. Compare mean official accuracy within each category, using equal weight for each of the five query subsets at K=8 and K=16. Category sizes refer to distinct questions, not the total number of predictions across runs.

For each condition, classify a question as improved, unchanged or worsened according to its official score relative to K=32. Also report exact agreement of the saved answer strings: identical scores do not necessarily imply identical predictions, and different strings may receive the same VQA score. The K-level summaries average the per-condition rates, not the signs of averaged scores. A small tolerance only avoids floating-point roundoff in score comparisons.

In [ ]:
answer_type_detail = pd.DataFrame(answer_type_rows).sort_values(["k", "seed", "answer_type"])
answer_type_summary = (
    answer_type_detail.groupby(["k", "answer_type"], as_index=False)
    .agg(mean_accuracy_pct=("vqa_accuracy_pct", "mean"), questions=("questions", "first"),
         runs=("condition", "nunique"))
)
answer_type_baseline = (answer_type_summary.loc[answer_type_summary.k == 32]
                        .set_index("answer_type")["mean_accuracy_pct"])
answer_type_summary["delta_vs_k32_pp"] = (
    answer_type_summary.mean_accuracy_pct - answer_type_summary.answer_type.map(answer_type_baseline)
)
if answer_type_summary.delta_vs_k32_pp.isna().any():
    raise ValueError("Answer-type comparison is missing a K=32 reference category.")

raw_answers = per_question.pivot(index="question_id", columns="condition", values="answer").reindex(question_ids)
if raw_answers.isna().any().any():
    raise ValueError("Missing saved answers in question-level comparisons.")

change_rows = []
for condition in conditions:
    name = condition["name"]
    delta = (scores[name] - baseline).to_numpy()
    unchanged = np.isclose(delta, 0, rtol=0, atol=1e-9)
    improved = (delta > 0) & ~unchanged
    worsened = (delta < 0) & ~unchanged
    exact_match = raw_answers[name].eq(raw_answers["k032"]).to_numpy()
    change_rows.append({
        "condition": name, "k": condition["k"], "seed": condition["seed"], "kind": condition["kind"],
        "questions": len(delta),
        "improved_count": int(improved.sum()), "unchanged_count": int(unchanged.sum()),
        "worsened_count": int(worsened.sum()), "improved_pct": 100 * improved.mean(),
        "unchanged_pct": 100 * unchanged.mean(), "worsened_pct": 100 * worsened.mean(),
        "exact_answer_agreement_pct": 100 * exact_match.mean(),
        "changed_answer_pct": 100 * (~exact_match).mean(),
        "delta_vs_k32_pp": delta.mean(),
    })
change_summary = pd.DataFrame(change_rows).sort_values(["k", "seed"])
change_group = (
    change_summary.groupby(["k", "kind"], as_index=False)
    .agg(runs=("condition", "nunique"), improved_pct=("improved_pct", "mean"),
         unchanged_pct=("unchanged_pct", "mean"), worsened_pct=("worsened_pct", "mean"),
         exact_answer_agreement_pct=("exact_answer_agreement_pct", "mean"),
         changed_answer_pct=("changed_answer_pct", "mean"), delta_vs_k32_pp=("delta_vs_k32_pp", "mean"))
)
assert np.allclose(change_group[["improved_pct", "unchanged_pct", "worsened_pct"]].sum(axis=1), 100)
assert np.allclose(change_group.delta_vs_k32_pp, group_summary.delta_vs_k32_pp)

display(answer_type_summary.round(3))
display(change_group.round(3))

## 6. Export tables and scoring protocol

All outputs are stored in `analysis/` inside the selected experiment directory. Existing filenames remain compatible with prior scoring notebooks. New CSVs include grouped answer-type accuracy, per-condition question-level score changes and grouped change rates. Rerunning this notebook replaces analysis artifacts for this run only, without touching raw prediction files.

The protocol documents the evaluator revision, bootstrap settings, recorded hardware, and interpretation of the additional metrics.

In [ ]:
report_dir = EXPERIMENT_DIR / "analysis"
report_dir.mkdir(exist_ok=True)
summary.to_csv(report_dir / "run_summary.csv", index=False)
per_question.to_csv(report_dir / "per_question_scores.csv", index=False)
comparisons.to_csv(report_dir / "paired_comparisons.csv", index=False)
group_summary.to_csv(report_dir / "group_summary.csv", index=False)
answer_type_detail.to_csv(report_dir / "accuracy_by_answer_type.csv", index=False)
answer_type_summary.to_csv(report_dir / "accuracy_by_answer_type_grouped.csv", index=False)
change_summary.to_csv(report_dir / "question_level_changes.csv", index=False)
change_group.to_csv(report_dir / "question_level_changes_grouped.csv", index=False)
(report_dir / "scoring_protocol.json").write_text(json.dumps({
    "experiment_signature": manifest["signature"], "evaluator_revision": EVALUATOR_REVISION,
    "evaluator_sha256": EVALUATOR_SHA256, "bootstrap_replicates": BOOTSTRAP_REPLICATES,
    "bootstrap_seed": BOOTSTRAP_SEED, "resampling_unit": "image",
    "group_interval_conditions_on": "the fixed query subsets", "score_units": "percent",
    "stage": experiment["stage"], "gpu_name": experiment["gpu_name"], "gpu_backend": backend,
    "interval_type": "pointwise percentile, 95%; not multiplicity-adjusted",
    "score_change_unit": "percentage points",
    "answer_agreement": "exact saved answer string equality",
    "group_change_rates": "arithmetic mean of per-condition percentages",
    "query_expansion": "repeated pretrained queries; not independently learned capacity",
}, indent=2) + "\n", encoding="utf-8")



## 7. Original summary figures

The accuracy plot uses sample SD across query subsets, not a confidence interval. The efficiency figure reports only the hardware recorded in the selected run. Generation throughput excludes preprocessing and other end-to-end overhead. The seed plot illustrates the sensitivity to query identity; seeds are paired across K=8 and K=16.

In [ ]:
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})
fig, ax = plt.subplots(figsize=(8, 5))
for name, ks, color, style in [
    ("Pruning and original baseline", [8, 16, 32], "#2166ac", "-"),
    ("Baseline and repeated-query controls", [32, 64, 128], "#b35806", "--"),
]:
    rows = group_summary[group_summary.k.isin(ks)]
    ax.errorbar(rows.k, rows.mean_accuracy_pct,
                yerr=rows.query_subset_accuracy_sd_pp.fillna(0),
                marker="o", linestyle=style, color=color, capsize=4, label=name)
ax.set(xlabel="Q-Former query count K", ylabel="VQA accuracy (%)",
       title=f"BLIP-2 query experiment ({experiment['stage']}, {len(question_ids)} questions)",
       xticks=[8, 16, 32, 64, 128])
ax.legend(fontsize=9)
fig.text(0.5, 0.01, "Error bars show SD across query subsets for K=8 and K=16.", ha="center", fontsize=9)
fig.tight_layout(rect=(0, 0.04, 1, 1))
fig.savefig(report_dir / "accuracy_vs_k.png", dpi=200)
plt.show()
plt.close(fig)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
colors = ["#2166ac", "#2166ac", "#555555", "#b35806", "#b35806"]
for ax, column, label in [
    (axes[0], "mean_median_latency_seconds", "Mean of per-run median latency (s)"),
    (axes[1], "mean_throughput_questions_per_second", "Mean generation throughput (questions/s)"),
    (axes[2], "max_peak_allocated_gib", "Maximum allocated GPU memory (GiB)"),
]:
    ax.bar(group_summary.k.astype(str), group_summary[column], color=colors)
    ax.set(xlabel="K", ylabel=label)
fig.suptitle(f"Generation measurements on {experiment['gpu_name']}")
fig.tight_layout()
fig.savefig(report_dir / "efficiency_vs_k.png", dpi=200)
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 4.5))
for k, color in [(8, "#2166ac"), (16, "#4393c3")]:
    rows = summary[summary.k == k]
    ax.plot(rows.seed, rows.vqa_accuracy_pct, marker="o", color=color, label=f"K={k}")
ax.axhline(float(summary.loc[summary.k == 32, "vqa_accuracy_pct"].iloc[0]),
           color="#555555", linestyle="--", label="K=32 baseline")
ax.set(xlabel="Query-selection seed", ylabel="VQA accuracy (%)",
       title="Sensitivity to retained query subsets", xticks=experiment["query_seeds"])
ax.legend()
fig.tight_layout()
fig.savefig(report_dir / "query_selection_variability.png", dpi=200)
plt.show()
plt.close(fig)



## 8. Additional figures

The paired-difference plot shows the 95% image-cluster bootstrap intervals for each K relative to K=32. Unlike the SD bars in the original accuracy figure, these intervals summarize uncertainty across evaluation images conditional on the fixed query subsets. Repeated-query controls are distinguished from pruning conditions.

In [ ]:
comparison_plot = group_summary.loc[group_summary.k != 32].copy()
fig, ax = plt.subplots(figsize=(8, 4.8))
for row in comparison_plot.itertuples():
    color = "#2166ac" if row.k < 32 else "#b35806"
    # Draw interval endpoints directly, even if the point lies outside a percentile CI.
    ax.hlines(str(row.k), row.paired_ci_lower_pp, row.paired_ci_upper_pp, color=color, linewidth=2)
    ax.plot([row.paired_ci_lower_pp, row.paired_ci_upper_pp], [str(row.k)] * 2,
            linestyle="none", marker="|", color=color, markersize=12)
    ax.scatter(row.delta_vs_k32_pp, str(row.k), color=color, s=45, zorder=3)
ax.axvline(0, linestyle="--", color="#555555", label="K=32 reference")
ax.invert_yaxis()
ax.set(xlabel="Accuracy difference vs. K=32 (percentage points)", ylabel="Query count K",
       title="Paired accuracy differences (95% image-bootstrap intervals)")
ax.legend(fontsize=9)
fig.text(0.5, 0.01, "Blue: pruning. Orange: repeated-query controls. Pointwise intervals, fixed subsets.", ha="center", fontsize=9)
fig.tight_layout(rect=(0, 0.04, 1, 1))
fig.savefig(report_dir / "paired_accuracy_differences.png", dpi=200)
plt.show()
plt.close(fig)

### Accuracy by answer type

The first panel reports absolute accuracy; the second panel shows differences from K=32 within each answer type. Missing categories in a small smoke sample are not invented. Counts are included in the legend so that small categories are not overinterpreted.

In [ ]:
answer_types = [a for a in ["yes/no", "number", "other"] if a in answer_type_summary.answer_type.unique()]
answer_types += sorted(set(answer_type_summary.answer_type) - set(answer_types))
category_counts = answer_type_summary.groupby("answer_type").questions.first()
accuracy_table = answer_type_summary.pivot(index="k", columns="answer_type", values="mean_accuracy_pct").reindex(columns=answer_types)
delta_table = answer_type_summary.pivot(index="k", columns="answer_type", values="delta_vs_k32_pp").reindex(columns=answer_types)
labels = {a: f"{a} (n={category_counts[a]:,})" for a in answer_types}
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
accuracy_table.rename(columns=labels).plot(kind="bar", ax=axes[0], rot=0)
delta_table.rename(columns=labels).plot(kind="bar", ax=axes[1], rot=0, legend=False)
axes[0].set(xlabel="K", ylabel="VQA accuracy (%)", title="Accuracy by answer type", ylim=(0, 100))
axes[1].set(xlabel="K", ylabel="Difference vs. K=32 (pp)", title="Accuracy change within answer type")
axes[1].axhline(0, color="#555555", linewidth=1)
axes[0].legend(title="Answer type", fontsize=9)
fig.tight_layout()
fig.savefig(report_dir / "accuracy_by_answer_type.png", dpi=200)
plt.show()
plt.close(fig)

### Question-level score changes and prediction agreement

The stacked bars summarize how often the VQA score improved, stayed unchanged, or worsened relative to K=32. The second panel reports the percentage of predictions whose saved answer strings exactly match K=32. The K=32 self-comparison is omitted from both panels. The group percentages average the runs at each K; individual seed results are retained in the CSV.

In [ ]:
changes_plot = change_group.loc[change_group.k != 32].set_index("k")
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
changes_plot[["improved_pct", "unchanged_pct", "worsened_pct"]].rename(columns={
    "improved_pct": "Improved", "unchanged_pct": "Unchanged", "worsened_pct": "Worsened"
}).plot(kind="bar", stacked=True, ax=axes[0], color=["#228833", "#bdbdbd", "#cc6677"], rot=0)
axes[0].set(xlabel="K", ylabel="Questions (%)", title="VQA score changes vs. K=32", ylim=(0, 100))
axes[0].legend(fontsize=9, loc="lower center", bbox_to_anchor=(0.5, -0.32), ncol=3)
axes[1].bar(changes_plot.index.astype(str), changes_plot.exact_answer_agreement_pct,
            color=["#2166ac" if k < 32 else "#b35806" for k in changes_plot.index])
axes[1].set(xlabel="K", ylabel="Exact answer agreement (%)", title="Literal prediction agreement vs. K=32", ylim=(0, 100))
fig.tight_layout(rect=(0, 0.05, 1, 1))
fig.savefig(report_dir / "question_level_changes.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"Saved scoring tables and six figures to: {report_dir}")

## 9. Interpretation and reporting

- Use the main-stage results for accuracy comparisons. The smoke stage is a functionality check; the pilot is exploratory.
- Report pruning separately from repeated-query expansion. K=64 and K=128 do not add independently learned visual representations.
- Distinguish between query-subset SD and paired image-bootstrap confidence intervals. Neither establishes an optimal query subset or a universal optimum for K.
- Category-level and question-level results are descriptive, and the answer-type sample sizes may differ. Exact answer agreement measures string equality, not semantic equivalence or correctness.
- Keep hardware-specific runs separate. Runtime and memory comparisons refer to the recorded GPU, software, decoding configuration and measured generation scope. Cross-device timing is not a controlled comparison.
- No LOQOA analysis is included in this notebook. The current report is based solely on the main query-count experiment.